In [ ]:
# Cell 1: 필수 라이브러리 및 Playwright 브라우저 설치
# 실행 환경에 Playwright와 pandas가 설치되어 있지 않다면 아래 셀을 먼저 실행하세요.
%pip install playwright pandas
!playwright install chromium


In [1]:
# Cell 2: 라이브러리 import 및 설정
import sys
import tempfile
import subprocess
import os
import json
import re
from urllib.parse import quote

# 노트북에서 시각 디버깅을 원하면 False로 설정하세요
NOTEBOOK_HEADLESS = True

# 검색할 키워드 입력 (여기를 수정하세요)
SEARCH_KEYWORD = "스마트폰"  # 예시: "여성 티셔츠", "노트북", "스마트폰" 등

# 최대 수집할 상품 개수
MAX_PRODUCTS = 30

# 검색 URL 생성
search_url = f"https://ssadagu.kr/shop/search.php?ss_tx={quote(SEARCH_KEYWORD)}"

print(f"🔍 검색 키워드: {SEARCH_KEYWORD}")
print(f"🔗 검색 URL: {search_url}")
print(f"📦 최대 수집 개수: {MAX_PRODUCTS}개\n")


🔍 검색 키워드: 스마트폰
🔗 검색 URL: https://ssadagu.kr/shop/search.php?ss_tx=%EC%8A%A4%EB%A7%88%ED%8A%B8%ED%8F%B0
📦 최대 수집 개수: 30개



In [ ]:
# Cell 3: 크롤링 스크립트 생성 함수

def create_ssadagu_crawl_script(headless, search_url, max_products, search_keyword):
    """
    크롤링 스크립트를 생성합니다.
    
    Args:
        headless: 헤드리스 모드 여부
        search_url: 검색 URL
        max_products: 최대 수집할 상품 개수
        search_keyword: 검색 키워드
        
    Returns:
        str: 크롤링 스크립트 문자열
    """
    script = f"""from playwright.sync_api import sync_playwright
import json
import time
import re

headless = {headless!r}
search_url = {search_url!r}
max_products = {max_products}
SEARCH_KEYWORD = {search_keyword!r}

products = []

with sync_playwright() as p:
    browser = p.chromium.launch(headless=headless)
    page = browser.new_page()
    
    print(f"접속 중: {{search_url}}")
    try:
        page.goto(search_url, wait_until="networkidle", timeout=30000)
        time.sleep(2)
        
        # 스크롤하여 동적 콘텐츠 로드 (더 많은 상품 로드)
        for scroll_idx in range(5):
            page.evaluate("window.scrollBy(0, window.innerHeight)")
            time.sleep(0.8)
        
        # 페이지 끝까지 스크롤
        page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        time.sleep(1)
        
        # 상품 리스트 찾기
        product_list = page.query_selector("ul.search_product_list")
        if not product_list:
            product_list = page.query_selector("#div_product_list")
        
        if product_list:
            # 각 상품 li 요소 찾기
            product_items = product_list.query_selector_all("li")
            print(f"\\n발견된 상품 수: {{len(product_items)}}개")
            
            # 유효한 상품이 max_products개가 될 때까지 더 많은 상품 처리
            processed_count = 0
            for item_elem in product_items:
                if len(products) >= max_products:
                    break
                processed_count += 1
                try:
                    # data 속성에서 정보 추출
                    title = item_elem.get_attribute("data-title") or ""
                    img_url = item_elem.get_attribute("data-img-url") or ""
                    
                    # 상품명 정제 (쉼표, 백슬래시, 큰따옴표 이후 불필요한 정보 제거)
                    if title:
                        # 큰따옴표로 둘러싸인 부분 제거 (이스케이프된 큰따옴표 포함)
                        title = re.sub(r'\\\\"[^"]*\\\\"', '', title).strip()
                        title = re.sub(r'"[^"]*"', '', title).strip()
                        # 백슬래시가 있으면 그 뒤 부분 제거
                        if "\\\\\\\\" in title:
                            title = title.split("\\\\\\\\")[0].strip()
                        # 쉼표로 분리하여 첫 번째 부분만 사용 (주요 상품명)
                        if "," in title:
                            title = title.split(",")[0].strip()
                        # 또는 특정 키워드 이후 제거
                        keywords_to_remove = [
                            "모든 네트워크 지원",
                            "심천 휴대폰 시장",
                            "공장 도매",
                            "인기 상품",
                            "도매",
                            "공장"
                        ]
                        for keyword in keywords_to_remove:
                            if keyword in title:
                                idx = title.find(keyword)
                                title = title[:idx].strip()
                                break
                    
                    # 상품 링크 찾기
                    product_link = ""
                    link_elem = item_elem.query_selector("a")
                    if link_elem:
                        href = link_elem.get_attribute("href") or ""
                        if href:
                            if href.startswith("http"):
                                product_link = href
                            else:
                                product_link = "https://ssadagu.kr" + href
                    
                    # 가격 정보 추출 (화면에 표시된 가격)
                    displayed_price = ""
                    try:
                        price_selectors = [
                            ".product_price",
                            "[class*='price']",
                            ".price",
                            "div.product_info"
                        ]
                        for selector in price_selectors:
                            price_elem = item_elem.query_selector(selector)
                            if price_elem:
                                price_text = price_elem.inner_text().strip()
                                # 숫자와 "원" 포함된 텍스트 추출
                                price_match = re.search(r'([\\d,]+)\\s*원', price_text)
                                if price_match:
                                    price_value_raw = price_match.group(1)
                                    price_value = price_value_raw.replace(",", "")
                                    # 숫자가 실제로 있는지 확인
                                    if price_value and price_value.isdigit():
                                        displayed_price = price_value_raw + "원"  # 원화 형식
                                        break
                    except:
                        pass
                    
                    # 최소한 제목이 있어야 유효한 상품
                    if title:
                        product_data = {{
                            "title": title,
                            "price": displayed_price,
                            "product_link": product_link,
                            "thumbnail_url": img_url
                        }}
                        products.append(product_data)
                        
                        # 목표 개수에 도달하면 중단
                        if len(products) >= max_products:
                            break
                except Exception as e:
                    print(f"  상품 정보 추출 오류: {{e}}")
                    continue
            
            if len(products) < max_products:
                print(f"\\n⚠️ 유효한 상품이 {{len(products)}}개만 수집되었습니다. (처리한 상품: {{processed_count}}개)")
        else:
            print("⚠ 상품 리스트를 찾을 수 없습니다.")
            
    except Exception as e:
        print(f"⚠ 크롤링 오류: {{e}}")
    
    browser.close()
    
    # 결과 저장
    result = {{
        "search_keyword": SEARCH_KEYWORD,
        "search_url": search_url,
        "total_products": len(products),
        "products": products
    }}
    
    # JSON 파일로 저장
    output_json = "ssadagu_search_results.json"
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    print(f"\\n✅ 총 {{len(products)}}개 상품 정보 수집 완료")
    print(f"✅ JSON 파일 저장 완료: {{output_json}}")
    
    # 결과 미리보기
    print(f"\\n📋 수집된 상품 미리보기 (처음 5개):")
    for idx, product in enumerate(products[:5], 1):
        print(f"\\n  {{idx}}. {{product['title'][:50]}}...")
        print(f"     가격: {{product['price']}}" if product['price'] else "     가격: 정보 없음")

        print(f"     링크: {{product['product_link'][:60]}}..." if product['product_link'] else "     링크: 정보 없음")
"""
    return script


In [ ]:
# Cell 4: 실행 및 결과 저장

print("▶ 키워드 기반 상품 검색을 시작합니다...\n")

# 크롤링 스크립트 생성
crawl_script = create_ssadagu_crawl_script(NOTEBOOK_HEADLESS, search_url, MAX_PRODUCTS, SEARCH_KEYWORD)

# 임시 파일에 스크립트 기록 후 실행
fd, path = tempfile.mkstemp(suffix="_crawl_by_keyword.py")
os.close(fd)
with open(path, "w", encoding="utf-8") as f:
    f.write(crawl_script)

try:
    completed = subprocess.run(
        [sys.executable, path], capture_output=True, text=True, check=False
    )
    print(completed.stdout)
    if completed.returncode != 0:
        print("--- 프로세스가 에러로 종료되었습니다 (stderr) ---")
        print(completed.stderr)
finally:
    try:
        os.remove(path)
    except Exception:
        pass


▶ 키워드 기반 상품 검색을 시작합니다...

접속 중: https://ssadagu.kr/shop/search.php?ss_tx=%EC%8A%A4%EB%A7%88%ED%8A%B8%ED%8F%B0

발견된 상품 수: 60개

✅ 총 30개 상품 정보 수집 완료
✅ JSON 파일 저장 완료: ssadagu_search_results.json

📋 수집된 상품 미리보기 (처음 5개):

  1. 2025 새로운 국경 스마트 폰 I16PROMax 안드로이드 전화 AliExpress 핫 ...
     가격: 42,042원
     링크: https://ssadagu.kr/shop/view.php?platform=1688&num_iid=90187...

  2. 안드로이드 5G 스마트 폰 i15 ProMax 게임 Netcom 듀얼 카드 P70ProMa...
     가격: 53,944원
     링크: https://ssadagu.kr/shop/view.php?platform=1688&num_iid=84744...

  3. 2025년형 풀 네트워크 신형 스마트 5G 휴대폰 16+512G i14Pro 고화질 대화면...
     가격: 75,075원
     링크: https://ssadagu.kr/shop/view.php?platform=1688&num_iid=77089...

  4. 국경 간 새로운 S24 스마트 안드로이드 폰 7.3 "올인원 16 1T HD 화면 재고 (...
     가격: 80,080원
     링크: https://ssadagu.kr/shop/view.php?platform=1688&num_iid=82817...

  5. 뜨거운 판매 6.8 인치 대형 화면 전체 netcom 5G 듀얼 카드 노인 기계 학생 저렴...
     가격: 19,778원
     링크: https://ssadagu.kr/shop/view.php?platform=1688&num_iid=83710...

